# ทำไม่ทันจริงๆ ครับ 555436463454365

พยายามทำ epochs น้อยๆ ให้ model ส่ง output ออกมาให้ไวที่สุด

# Requirement & Setup

In [ ]:
!pip install tiktoken

In [ ]:
!pip install sentencepiece protobuf

In [ ]:
import os
import json
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from tqdm import tqdm

In [ ]:
TRAIN_JSON = r"C:\Users\narak\Downloads\sup_ai\image_cap\capgen_v1.0_train.json"
TRAIN_DIR = r"C:\Users\narak\Downloads\sup_ai\image_cap\train\train"
TEST_DIR = r"C:\Users\narak\Downloads\sup_ai\image_cap\test\test"
SAMPLE_SUB = r"C:\Users\narak\Downloads\sup_ai\image_cap\sample_submission.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# model

In [ ]:
encoder_model = "google/vit-base-patch16-224-in21k"
decoder_model = "airesearch/wangchanberta-base-att-spm-uncased"

feature_extractor = ViTImageProcessor.from_pretrained(encoder_model)
tokenizer = AutoTokenizer.from_pretrained(decoder_model)

model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    encoder_model, decoder_model
).to(DEVICE)

model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# data

In [ ]:
class ThaiCapDataset(Dataset):
    def __init__(self, json_path, img_dir, processor, tokenizer, max_samples=500):
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if "root" in data:
                data = data["root"]
        
        self.processor = processor
        self.tokenizer = tokenizer
        self.samples = []
        
        available_images = {}
        for root_dir, _, files in os.walk(img_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    available_images[file] = os.path.join(root_dir, file)
        
        missing_count = 0
        
        for json_img_path, captions in data.items():
            if max_samples is not None and len(self.samples) >= max_samples:
                break
                
            filename = os.path.basename(json_img_path)
            
            if filename in available_images:
                full_path = available_images[filename]
                for cap in captions:
                    self.samples.append({"img_path": full_path, "caption": cap})
            else:
                missing_count += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()
        
        labels = self.tokenizer(
            item["caption"], max_length=32, padding="max_length", truncation=True, return_tensors="pt"
        ).input_ids.squeeze()
        
        labels[labels == self.tokenizer.pad_token_id] = -100 
        
        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
train_dataset = ThaiCapDataset(TRAIN_JSON, TRAIN_DIR, feature_extractor, tokenizer, max_samples=500)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

for batch in tqdm(train_loader, desc="Training Epoch 1"):
    pixel_values = batch["pixel_values"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)
    
    outputs = model(pixel_values=pixel_values, labels=labels)
    loss = outputs.loss
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# submission / inference

In [ ]:
model.eval()
sub_df = pd.read_csv(SAMPLE_SUB)
predictions = []

test_images_paths = {}
for root_dir, _, files in os.walk(TEST_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_images_paths[file] = os.path.join(root_dir, file)

img_id_col = sub_df.columns[0] 

for img_name in tqdm(sub_df[img_id_col], desc="Generating Captions"):
    filename = os.path.basename(str(img_name))
    
    if filename in test_images_paths:
        img_path = test_images_paths[filename]
        image = Image.open(img_path).convert("RGB")
        pixel_values = feature_extractor(images=image, return_tensors="pt").pixel_values.to(DEVICE)
        with torch.no_grad():
            output_ids = model.generate(pixel_values, max_length=32, num_beams=3)
        caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        predictions.append(caption)
    else:
        predictions.append("รูปภาพผู้คนและสิ่งของ")

caption_col = sub_df.columns[1] if len(sub_df.columns) > 1 else 'caption'
sub_df[caption_col] = predictions

output_csv = "submission_v2.csv"
sub_df.to_csv(output_csv, index=False)